In [1]:
import pandas as pd
import numpy as np


In [2]:
import matplotlib
matplotlib.use('Agg')

In [3]:
import matplotlib.pyplot as plt

In [4]:
import seaborn as sns
import warnings
import os

In [5]:
from sklearn.model_selection import StratifiedKFold

In [6]:
from sklearn.metrics import classification_report ,f1_score,balanced_accuracy_score
from sklearn.metrics import confusion_matrix,cohen_kappa_score

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack,csr_matrix

In [8]:
import lightgbm as lgb
import xgboost as xgb

In [9]:
warnings.filterwarnings('ignore')

In [10]:
sns.set_theme(style ='whitegrid',palette = 'muted')

In [11]:
SEED = 42
np.random.seed(SEED)

In [12]:
print("=" * 70)
print("🏥 TRIAGEGEIST: Ultimate AI Emergency Triage Solution")
print("=" * 70)

🏥 TRIAGEGEIST: Ultimate AI Emergency Triage Solution


In [13]:
print("\n📋 SECTION 1: DATA LOADING")
print("-" * 50)


📋 SECTION 1: DATA LOADING
--------------------------------------------------


In [14]:
Base = '/kaggle/input/competitions/triagegeist'
train = pd.read_csv(f'{Base}/train.csv')
test = pd.read_csv(f'{Base}/test.csv')
cc = pd.read_csv(f'{Base}/chief_complaints.csv')
ph = pd.read_csv(f'{Base}/patient_history.csv')
ss = pd.read_csv(f'{Base}/sample_submission.csv')

In [15]:
print(f"  Train:             {train.shape[0]:>6,} rows × {train.shape[1]} cols")
print(f"  Test:              {test.shape[0]:>6,} rows × {test.shape[1]} cols")
print(f"  Chief Complaints:  {cc.shape[0]:>6,} rows × {cc.shape[1]} cols")
print(f"  Patient History:   {ph.shape[0]:>6,} rows × {ph.shape[1]} cols")

  Train:             80,000 rows × 40 cols
  Test:              20,000 rows × 37 cols
  Chief Complaints:  100,000 rows × 3 cols
  Patient History:   100,000 rows × 26 cols


In [16]:
train = train.merge(cc[['patient_id','chief_complaint_raw']],on = 'patient_id', how='left')

In [17]:
train = train.merge(ph,on = 'patient_id',how='left')

In [18]:
test = test.merge(cc[['patient_id','chief_complaint_raw']],on = 'patient_id', how='left')

In [19]:
test = test.merge(ph,on = 'patient_id',how='left')

In [20]:
train.shape

(80000, 66)

In [21]:
test.shape

(20000, 63)

In [22]:
Target = 'triage_acuity'
y_train = train[Target].values

In [23]:
y_train

array([2, 5, 5, ..., 5, 1, 4])

In [24]:
print(f"\n  Target distribution:")



  Target distribution:


In [25]:
sorted(train[Target].unique())

[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

In [26]:
for level in sorted(train[Target].unique()):
    n= (y_train == level).sum()
    pct = n/len(y_train)*100
    labels = {1: "Resuscitation", 2: "Emergent", 3: "Urgent",
              4: "Less Urgent", 5: "Non-Urgent"}
    bar = "█" * int(pct / 2)
    print(f"    ESI {level} ({labels.get(level, '')}): {n:>6,}({pct:>5.1f}%) {bar}")

    ESI 1 (Resuscitation):  3,222(  4.0%) ██
    ESI 2 (Emergent): 13,439( 16.8%) ████████
    ESI 3 (Urgent): 28,921( 36.2%) ██████████████████
    ESI 4 (Less Urgent): 23,020( 28.8%) ██████████████
    ESI 5 (Non-Urgent): 11,398( 14.2%) ███████


In [27]:
print("\n\n📊 SECTION 2: EXPLORATORY DATA ANALYSIS")
print("-" * 50)



📊 SECTION 2: EXPLORATORY DATA ANALYSIS
--------------------------------------------------


In [28]:
fig,axes = plt.subplots(2,3,figsize=(18,11))
fig.suptitle('Triagegeist — Emergency Department Triage EDA', fontsize = 16,fontweight = 'bold')
colors_esi = ['#d32f2f', '#f57c00', '#fbc02d', '#66bb6a', '#42a5f5']
plt.tight_layout()


In [29]:
ax = axes[0,0]
dist = train[Target].value_counts().sort_index()
dist.plot(kind='bar', ax=ax, color=colors_esi, edgecolor='black', alpha=0.85)
ax.set_title('Triage Acuity Distribution', fontweight='bold')
ax.set_xlabel('ESI Level')
ax.set_ylabel('Count')
ax.set_xticklabels(['1\nResus', '2\nEmerg', '3\nUrgent', '4\nLess\nUrg', '5\nNon\nUrg'], rotation=0)


[Text(0, 0, '1\nResus'),
 Text(1, 0, '2\nEmerg'),
 Text(2, 0, '3\nUrgent'),
 Text(3, 0, '4\nLess\nUrg'),
 Text(4, 0, '5\nNon\nUrg')]

In [30]:
ax = axes[0, 1]
data_hr = [train[train[Target] == l]['heart_rate'].dropna() for l in range(1, 6)]
bp = ax.boxplot(data_hr, labels=[f'ESI {l}' for l in range(1, 6)], patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], colors_esi):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax.set_title('Heart Rate by Triage Level', fontweight='bold')
ax.set_ylabel('BPM')

Text(606.4444444444446, 0.5, 'BPM')

In [31]:
ax = axes[0, 2]
data_spo2 = [train[train[Target] == l]['spo2'].dropna() for l in range(1, 6)]
bp2 = ax.boxplot(data_spo2, labels=[f'ESI {l}' for l in range(1, 6)], patch_artist=True, showfliers=False)
for patch, color in zip(bp2['boxes'], colors_esi):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax.set_title('Oxygen Saturation by Triage Level', fontweight='bold')
ax.set_ylabel('SpO₂ (%)')

Text(1200.4444444444446, 0.5, 'SpO₂ (%)')

In [32]:
ax = axes[1, 0]
data_news = [train[train[Target] == l]['news2_score'].dropna() for l in range(1, 6)]
bp3 = ax.boxplot(data_news, labels=[f'ESI {l}' for l in range(1, 6)], patch_artist=True, showfliers=False)
for patch, color in zip(bp3['boxes'], colors_esi):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax.set_title('NEWS2 Score by Triage Level', fontweight='bold')
ax.set_ylabel('NEWS2')

Text(12.444444444444452, 0.5, 'NEWS2')

In [33]:
ax = axes[1, 1]
for i, level in enumerate(range(1, 6)):
    subset = train[train[Target] == level]['age']
    ax.hist(subset, bins=30, alpha=0.4, label=f'ESI {level}', color=colors_esi[i])
ax.set_title('Age Distribution by Triage Level', fontweight='bold')
ax.set_xlabel('Age')
ax.legend(fontsize=8)

In [34]:
ax = axes[1, 2]
cc_counts = train.groupby(['chief_complaint_system', Target]).size().unstack(fill_value=0)
cc_pcts = cc_counts.div(cc_counts.sum(axis=1), axis=0)
top_cc = cc_counts.sum(axis=1).nlargest(8).index
cc_pcts.loc[top_cc].plot(kind='barh', stacked=True, ax=ax, color=colors_esi, alpha=0.85)
ax.set_title('Acuity Distribution by Complaint System', fontweight='bold')
ax.set_xlabel('Proportion')
ax.legend(title='ESI', fontsize=7)



In [35]:
plt.tight_layout()
plt.savefig('/kaggle/working/eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("  ✅ EDA plots saved")

  ✅ EDA plots saved


In [36]:
print("\n\n⚙️ SECTION 3: FEATURE ENGINEERING")
print("-" * 50)



⚙️ SECTION 3: FEATURE ENGINEERING
--------------------------------------------------


In [37]:
LEAK_COLS = ['disposition','ed_los_hours']
ID_COLS = ['patient_id','site_id','triage_nurse_id']
DROP_COLS = LEAK_COLS + ID_COLS +[Target + 'chief_complaint_raw']
print(f"  ⚠️ Removing leakage columns: {LEAK_COLS}")
print(f"  ⚠️ Removing ID columns: {ID_COLS}")

  ⚠️ Removing leakage columns: ['disposition', 'ed_los_hours']
  ⚠️ Removing ID columns: ['patient_id', 'site_id', 'triage_nurse_id']


In [38]:
cat_cols = train.select_dtypes(include = 'object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in DROP_COLS]
print(f"  📝 Categorical columns to encode: {cat_cols}")

  📝 Categorical columns to encode: ['arrival_mode', 'arrival_day', 'arrival_season', 'shift', 'age_group', 'sex', 'language', 'insurance_type', 'transport_origin', 'pain_location', 'mental_status_triage', 'chief_complaint_system', 'chief_complaint_raw']


In [39]:
from sklearn.preprocessing import LabelEncoder
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train[col],test[col]],axis = 0).astype(str)
    le.fit(combined)
    train[col + '_enc'] = le.transform(train[col].astype(str))
    test[col+'_enc'] = le.transform(test[col].astype(str))
    label_encoders[col] = le
print(f"  ✅ Label encoded {len(cat_cols)} categorical features")

  ✅ Label encoded 13 categorical features


In [40]:
for df in [train, test]:
    # Vital sign abnormality flags
    df['hypoxia'] = (df['spo2'] < 92).astype(int)
    df['severe_hypoxia'] = (df['spo2'] < 88).astype(int)
    df['fever'] = (df['temperature_c'] > 38.0).astype(int)
    df['hypothermia'] = (df['temperature_c'] < 35.5).astype(int)
    df['tachycardia'] = (df['heart_rate'] > 100).astype(int)
    df['bradycardia'] = (df['heart_rate'] < 50).astype(int)
    df['tachypnea'] = (df['respiratory_rate'] > 22).astype(int)
    df['hypotension'] = (df['systolic_bp'] < 90).astype(int)
    df['hypertensive_crisis'] = (df['systolic_bp'] > 180).astype(int)
    df['altered_mental'] = (df['gcs_total'] < 15).astype(int)
    df['severe_pain'] = (df['pain_score'] >= 8).astype(int)
    df['high_shock_index'] = (df['shock_index'] > 1.0).astype(int)

    
    flag_cols = ['hypoxia', 'severe_hypoxia', 'fever', 'hypothermia', 'tachycardia',
                 'bradycardia', 'tachypnea', 'hypotension', 'hypertensive_crisis',
                 'altered_mental', 'severe_pain', 'high_shock_index']
    df['n_abnormal_vitals'] = df[flag_cols].sum(axis=1)
    
    hx_cols = [c for c in df.columns if c.startswith('hx_')]
    df['comorbidity_burden'] = df[hx_cols].sum(axis=1)
    
    # Interaction features
    df['age_x_comorbidities'] = df['age'] * df['num_comorbidities']
    df['shock_x_altered'] = df['shock_index'] * df['altered_mental']
    df['news2_x_age'] = df['news2_score'] * (df['age'] / 100)

print(f"  ✅ Created clinical flag features and interactions")



  ✅ Created clinical flag features and interactions


In [41]:
print(f"\n  📝 NLP: Extracting TF-IDF features from chief complaints...")
train['chief_complaint_raw'] = train['chief_complaint_raw'].fillna('')
test['chief_complaint_raw'] = test['chief_complaint_raw'].fillna('')


  📝 NLP: Extracting TF-IDF features from chief complaints...


In [42]:
tfidf = TfidfVectorizer(
    max_features = 100,
    ngram_range = (1,2),
    min_df = 5,
    stop_words = 'english',
    strip_accents = 'unicode'
    
)

In [43]:
tfidf_train = tfidf.fit_transform(train['chief_complaint_raw'])
tfidf_test = tfidf.fit_transform(test['chief_complaint_raw'])
print(f"  ✅ TF-IDF: {tfidf_train.shape[1]} features from {len(tfidf.vocabulary_)} terms")

  ✅ TF-IDF: 100 features from 100 terms


In [44]:
# ---- ASSEMBLE FINAL FEATURE MATRIX ----
numeric_cols = [c for c in train.columns if c not in DROP_COLS
                and (train[c].dtype in ['int64', 'float64', 'int32', 'float32', 'uint8'])]

In [45]:
# Add encoded categoricals
enc_cols = [c + '_enc' for c in cat_cols]
feature_cols = [c for c in numeric_cols + enc_cols if c in train.columns and c != Target]

In [46]:
# Remove original cat cols from feature_cols
feature_cols = [c for c in feature_cols if c not in cat_cols]

In [47]:
print(f"\n  Final tabular features: {len(feature_cols)}")
print(f"  TF-IDF features: {tfidf_train.shape[1]}")
print(f"  Total features: {len(feature_cols) + tfidf_train.shape[1]}")

X_tab_train = train[feature_cols].values.astype(np.float32)
X_tab_test = test[feature_cols].values.astype(np.float32)


  Final tabular features: 90
  TF-IDF features: 100
  Total features: 190


In [48]:
# Combine tabular + TF-IDF
X_train_full = np.hstack([X_tab_train, tfidf_train.toarray().astype(np.float32)])
X_test_full = np.hstack([X_tab_test, tfidf_test.toarray().astype(np.float32)])

In [49]:
all_feature_names = feature_cols + [f'tfidf_{w}' for w in tfidf.get_feature_names_out()]
print(f"\n  ✅ Feature matrices: Train {X_train_full.shape}, Test {X_test_full.shape}")


  ✅ Feature matrices: Train (80000, 190), Test (20000, 190)


In [50]:
print("\n\n🤖 SECTION 4: MODEL TRAINING (LightGBM + XGBoost Ensemble)")
print("-" * 50)



🤖 SECTION 4: MODEL TRAINING (LightGBM + XGBoost Ensemble)
--------------------------------------------------


In [51]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

In [52]:
print("\n  🌿 Training LightGBM...")
lgb_params = {
    'objective': 'multiclass', 'num_class': 5, 'metric': 'multi_logloss',
    'boosting_type': 'gbdt', 'num_leaves': 127, 'learning_rate': 0.05,
    'feature_fraction': 0.75, 'bagging_fraction': 0.75, 'bagging_freq': 5,
    'min_child_samples': 30, 'n_estimators': 1000, 'random_state': SEED,
    'verbose': -1, 'class_weight': 'balanced',
}


  🌿 Training LightGBM...


In [53]:
lgb_oof = np.zeros((len(X_train_full), 5))
lgb_test_preds = np.zeros((len(X_test_full), 5))
lgb_models = []

In [54]:
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_full, y_train)):
    X_tr, X_val = X_train_full[tr_idx], X_train_full[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr - 1, eval_set=[(X_val, y_val - 1)],
              callbacks=[lgb.early_stopping(50, verbose=False)])

    lgb_oof[val_idx] = model.predict_proba(X_val)
    lgb_test_preds += model.predict_proba(X_test_full) / N_FOLDS
    lgb_models.append(model)

    f1 = f1_score(y_val, lgb_oof[val_idx].argmax(1) + 1, average='macro')
    print(f"    Fold {fold+1}: F1={f1:.4f}")

    Fold 1: F1=0.9924
    Fold 2: F1=0.9921
    Fold 3: F1=0.9944
    Fold 4: F1=0.9927
    Fold 5: F1=0.9928


In [55]:
lgb_f1 = f1_score(y_train, lgb_oof.argmax(1) + 1, average='macro')
print(f"  LightGBM OOF Macro F1: {lgb_f1:.4f}")

  LightGBM OOF Macro F1: 0.9929


In [56]:
print("\n  🚀 Training XGBoost...")
xgb_params = {
    'objective': 'multi:softprob', 'num_class': 5, 'eval_metric': 'mlogloss',
    'max_depth': 8, 'learning_rate': 0.05, 'subsample': 0.75,
    'colsample_bytree': 0.75, 'min_child_weight': 30,
    'n_estimators': 1000, 'random_state': SEED, 'verbosity': 0,
    'tree_method': 'hist',
}


  🚀 Training XGBoost...


In [57]:
xgb_oof = np.zeros((len(X_train_full), 5))
xgb_test_preds = np.zeros((len(X_test_full), 5))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_full, y_train)):
    X_tr, X_val = X_train_full[tr_idx], X_train_full[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_tr, y_tr - 1, eval_set=[(X_val, y_val - 1)], verbose=False)

    xgb_oof[val_idx] = model.predict_proba(X_val)
    xgb_test_preds += model.predict_proba(X_test_full) / N_FOLDS

    f1 = f1_score(y_val, xgb_oof[val_idx].argmax(1) + 1, average='macro')
    print(f"    Fold {fold+1}: F1={f1:.4f}")

xgb_f1 = f1_score(y_train, xgb_oof.argmax(1) + 1, average='macro')
print(f"  XGBoost OOF Macro F1: {xgb_f1:.4f}")

    Fold 1: F1=0.9870
    Fold 2: F1=0.9870
    Fold 3: F1=0.9902
    Fold 4: F1=0.9878
    Fold 5: F1=0.9898
  XGBoost OOF Macro F1: 0.9884


In [58]:
print("\n  🎯 Weighted Ensemble...")
# Optimize weight
best_w, best_f1 = 0.5, 0
for w in np.arange(0.3, 0.8, 0.05):
    ens = w * lgb_oof + (1 - w) * xgb_oof
    f1 = f1_score(y_train, ens.argmax(1) + 1, average='macro')
    if f1 > best_f1:
        best_w, best_f1 = w, f1

print(f"  Optimal weight: LGB={best_w:.2f}, XGB={1-best_w:.2f}")


  🎯 Weighted Ensemble...
  Optimal weight: LGB=0.75, XGB=0.25


In [59]:
ens_oof = best_w * lgb_oof + (1 - best_w) * xgb_oof
ens_test = best_w * lgb_test_preds + (1 - best_w) * xgb_test_preds

y_oof_pred = ens_oof.argmax(1) + 1
y_test_pred = ens_test.argmax(1) + 1

ens_f1 = f1_score(y_train, y_oof_pred, average='macro')
ens_ba = balanced_accuracy_score(y_train, y_oof_pred)
ens_kappa = cohen_kappa_score(y_train, y_oof_pred, weights='quadratic')

In [60]:
print(f"\n  📊 ENSEMBLE RESULTS (OOF):")
print(f"     Macro F1-Score:     {ens_f1:.4f}")
print(f"     Balanced Accuracy:  {ens_ba:.4f}")
print(f"     Quadratic Kappa:    {ens_kappa:.4f}")
print(f"\n  📋 Classification Report:")
names = ['ESI 1 (Resus)', 'ESI 2 (Emerg)', 'ESI 3 (Urgent)', 'ESI 4 (Less Urg)', 'ESI 5 (Non-Urg)']
present = sorted(np.unique(y_train))
print(classification_report(y_train, y_oof_pred, labels=present,
                            target_names=[names[c-1] for c in present]))


  📊 ENSEMBLE RESULTS (OOF):
     Macro F1-Score:     0.9927
     Balanced Accuracy:  0.9915
     Quadratic Kappa:    0.9986

  📋 Classification Report:
                  precision    recall  f1-score   support

   ESI 1 (Resus)       0.98      0.97      0.97      3222
   ESI 2 (Emerg)       0.99      0.99      0.99     13439
  ESI 3 (Urgent)       1.00      1.00      1.00     28921
ESI 4 (Less Urg)       1.00      1.00      1.00     23020
 ESI 5 (Non-Urg)       1.00      1.00      1.00     11398

        accuracy                           1.00     80000
       macro avg       0.99      0.99      0.99     80000
    weighted avg       1.00      1.00      1.00     80000



In [61]:
print("\n📊 SECTION 5: MODEL INTERPRETATION")
print("-" * 50)

importances = np.zeros(len(all_feature_names))
for m in lgb_models:
    importances += m.feature_importances_
importances /= len(lgb_models)


📊 SECTION 5: MODEL INTERPRETATION
--------------------------------------------------


In [62]:
feat_imp = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=False).head(25)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Model Interpretation — Feature Importance & Confusion Matrix', fontsize=15, fontweight='bold')

Text(0.5, 0.98, 'Model Interpretation — Feature Importance & Confusion Matrix')

In [63]:
ax = axes[0]
ax.barh(range(len(feat_imp)), feat_imp['importance'].values, color='steelblue', edgecolor='navy', alpha=0.8)
ax.set_yticks(range(len(feat_imp)))
ax.set_yticklabels(feat_imp['feature'].values, fontsize=9)
ax.set_xlabel('Mean Importance (Gain)')
ax.set_title('Top 25 Features — LightGBM', fontweight='bold')
ax.invert_yaxis()

In [64]:
ax = axes[1]
cm = confusion_matrix(y_train, y_oof_pred, labels=present)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            xticklabels=[f'ESI {c}' for c in present],
            yticklabels=[f'ESI {c}' for c in present])
ax.set_title('Normalized Confusion Matrix', fontweight='bold')
ax.set_ylabel('True'); ax.set_xlabel('Predicted')

Text(0.5, 54.249999999999986, 'Predicted')

In [65]:
plt.tight_layout()
plt.savefig('/kaggle/working/model_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [66]:
print("\n  🔑 KEY FINDINGS:")
for i, row in feat_imp.head(10).iterrows():
    print(f"     {feat_imp.index.get_loc(i)+1}. {row['feature']} (importance: {row['importance']:.0f})")


  🔑 KEY FINDINGS:
     1. chief_complaint_raw_enc (importance: 15757)
     2. chief_complaint_system_enc (importance: 6676)
     3. chief_complaint_raw_enc (importance: 4582)
     4. respiratory_rate (importance: 3992)
     5. spo2 (importance: 3981)
     6. temperature_c (importance: 3018)
     7. mean_arterial_pressure (importance: 2835)
     8. pain_score (importance: 2501)
     9. news2_score (importance: 2364)
     10. heart_rate (importance: 2355)


In [67]:
print("\n\n⚖️ SECTION 6: TRIAGE BIAS AND FAIRNESS ANALYSIS")
print("-" * 50)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Triage Bias Analysis — Identifying Systematic Disparities', fontsize=14, fontweight='bold')



⚖️ SECTION 6: TRIAGE BIAS AND FAIRNESS ANALYSIS
--------------------------------------------------


Text(0.5, 0.98, 'Triage Bias Analysis — Identifying Systematic Disparities')

In [68]:
ax = axes[0, 0]
f1_by_sex = {}
for sex_val in train['sex'].unique():
    mask = train['sex'].values == sex_val
    if mask.sum() > 100:
        f1 = f1_score(y_train[mask], y_oof_pred[mask], average='macro')
        f1_by_sex[sex_val] = f1
        print(f"  Sex={sex_val}: F1={f1:.4f} (n={mask.sum():,})")
ax.bar(f1_by_sex.keys(), f1_by_sex.values(), color=['#42a5f5', '#ef5350', '#66bb6a'],
       edgecolor='black', alpha=0.8)
ax.set_title('Model Performance by Sex', fontweight='bold')
ax.set_ylabel('Macro F1'); ax.set_ylim(0, 1)
for i, (k, v) in enumerate(f1_by_sex.items()):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

  Sex=M: F1=0.9915 (n=37,735)
  Sex=F: F1=0.9939 (n=40,339)
  Sex=Other: F1=0.9912 (n=1,926)


In [69]:
ax = axes[0, 1]
f1_by_age = {}
for grp in train['age_group'].unique():
    mask = train['age_group'].values == grp
    if mask.sum() > 100:
        f1 = f1_score(y_train[mask], y_oof_pred[mask], average='macro')
        f1_by_age[grp] = f1
        print(f"  Age={grp}: F1={f1:.4f} (n={mask.sum():,})")
ax.bar(f1_by_age.keys(), f1_by_age.values(), color='#66bb6a', edgecolor='black', alpha=0.8)
ax.set_title('Performance by Age Group', fontweight='bold')
ax.set_ylabel('Macro F1'); ax.set_ylim(0, 1)
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
for i, (k, v) in enumerate(f1_by_age.items()):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold', fontsize=8)

  Age=middle_aged: F1=0.9929 (n=27,889)
  Age=elderly: F1=0.9927 (n=21,653)
  Age=young_adult: F1=0.9922 (n=23,863)
  Age=pediatric: F1=0.9937 (n=6,595)


In [70]:
# 6c. Undertriage analysis by age group
ax = axes[1, 0]
undertriage_rates = {}
for grp in train['age_group'].unique():
    mask = train['age_group'].values == grp
    if mask.sum() > 100:
        ut_rate = (y_oof_pred[mask] > y_train[mask]).mean()
        undertriage_rates[grp] = ut_rate
ax.bar(undertriage_rates.keys(), undertriage_rates.values(), color='#d32f2f', edgecolor='black', alpha=0.8)
ax.set_title('Under-triage Rate by Age Group', fontweight='bold')
ax.set_ylabel('Under-triage Proportion')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
for i, (k, v) in enumerate(undertriage_rates.items()):
    ax.text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold', fontsize=8)

In [71]:
ax = axes[1, 1]
undertriage = (y_oof_pred > y_train).mean()
overtriage = (y_oof_pred < y_train).mean()
correct = (y_oof_pred == y_train).mean()
rates = [correct, overtriage, undertriage]
labels_err = ['Correct\nTriage', 'Over-triage\n(safer)', 'Under-triage\n(dangerous)']
colors_err = ['#66bb6a', '#fbc02d', '#d32f2f']
ax.bar(labels_err, rates, color=colors_err, edgecolor='black', alpha=0.85)
ax.set_title('Triage Error Direction', fontweight='bold')
ax.set_ylabel('Proportion')
for i, v in enumerate(rates):
    ax.text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('/kaggle/working/bias_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [72]:

print(f"\n  ⚠️ SAFETY SUMMARY:")
print(f"     Correct triage:   {correct:.1%}")
print(f"     Over-triage:      {overtriage:.1%} (errs toward safety)")
print(f"     Under-triage:     {undertriage:.1%} (DANGEROUS — delays care)")


  ⚠️ SAFETY SUMMARY:
     Correct triage:   99.7%
     Over-triage:      0.1% (errs toward safety)
     Under-triage:     0.2% (DANGEROUS — delays care)


In [73]:
print("\n\n📝 SECTION 7: NLP INSIGHTS FROM CHIEF COMPLAINTS")
print("-" * 50)



📝 SECTION 7: NLP INSIGHTS FROM CHIEF COMPLAINTS
--------------------------------------------------


In [74]:
# Most common complaint words by acuity
print("\n  Top chief complaint terms by ESI level:")
for level in [1, 2, 5]:
    mask = y_train == level
    texts = train.loc[mask, 'chief_complaint_raw'].fillna('')
    # Simple word frequency
    from collections import Counter
    words = Counter()
    for t in texts:
        for w in t.lower().split(', '):
            w = w.strip()
            if len(w) > 3:
                words[w] += 1
    top5 = words.most_common(5)
    print(f"    ESI {level}: {', '.join([f'{w[0]}({w[1]})' for w in top5])}")


  Top chief complaint terms by ESI level:
    ESI 1: for 2 days(221), since yesterday(221), no prior history of this(211), intermittent(206), worsening with movement(205)
    ESI 2: since yesterday(942), no prior history of this(903), intermittent(897), worsening with movement(882), constant(880)
    ESI 5: since yesterday(800), for 2 days(778), worsening with movement(755), no prior history of this(751), onset today(730)


In [75]:
print("\n\n💡 SECTION 8: CLINICAL DECISION SUPPORT DEMO")
print("-" * 50)

esi_labels = {1: "🔴 RESUSCITATION", 2: "🟠 EMERGENT", 3: "🟡 URGENT",
              4: "🔵 LESS URGENT", 5: "🟢 NON-URGENT"}

# Show confident vs uncertain predictions
max_probs = ens_oof.max(axis=1)
confident = max_probs > 0.7
uncertain = max_probs < 0.4



💡 SECTION 8: CLINICAL DECISION SUPPORT DEMO
--------------------------------------------------


In [76]:
print(f"\n  Prediction confidence analysis:")
print(f"    High confidence (>70%): {confident.sum():,} ({confident.mean():.1%}) — model is sure")
print(f"    Low confidence (<40%):  {uncertain.sum():,} ({uncertain.mean():.1%}) — needs nurse review")
print(f"    Medium confidence:      {(~confident & ~uncertain).sum():,} ({(~confident & ~uncertain).mean():.1%})")


  Prediction confidence analysis:
    High confidence (>70%): 79,763 (99.7%) — model is sure
    Low confidence (<40%):  0 (0.0%) — needs nurse review
    Medium confidence:      237 (0.3%)


In [77]:
# F1 for high-confidence predictions only
if confident.sum() > 100:
    f1_conf = f1_score(y_train[confident], y_oof_pred[confident], average='macro')
    print(f"\n    F1 on high-confidence subset: {f1_conf:.4f}")
    print(f"    → AI is most accurate when it's most confident — a natural")
    print(f"      threshold for selective automation in triage workflows.")


    F1 on high-confidence subset: 0.9950
    → AI is most accurate when it's most confident — a natural
      threshold for selective automation in triage workflows.


In [78]:
'''
print("\n\n📤 SECTION 9: GENERATING SUBMISSION")
print("-" * 50)

submission = pd.DataFrame({
    'patient_id': test['patient_id'],
    'triage_acuity': y_test_pred
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f"  ✅ submission.csv saved: {submission.shape}")
print(f"  Prediction distribution:")
for level in sorted(submission['triage_acuity'].unique()):
    n = (submission['triage_acuity'] == level).sum()
    print(f"    ESI {level}: {n:>5,} ({n/len(submission)*100:.1f}%)")
'''


'\nprint("\n\n📤 SECTION 9: GENERATING SUBMISSION")\nprint("-" * 50)\n\nsubmission = pd.DataFrame({\n    \'patient_id\': test[\'patient_id\'],\n    \'triage_acuity\': y_test_pred\n})\nsubmission.to_csv(\'/kaggle/working/submission.csv\', index=False)\nprint(f"  ✅ submission.csv saved: {submission.shape}")\nprint(f"  Prediction distribution:")\nfor level in sorted(submission[\'triage_acuity\'].unique()):\n    n = (submission[\'triage_acuity\'] == level).sum()\n    print(f"    ESI {level}: {n:>5,} ({n/len(submission)*100:.1f}%)")\n'